In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim

In [19]:
df = pd.read_csv("fmnist_small.csv")
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0


In [20]:
x = df.iloc[:, 1:]
y = df.iloc[:, 0]

In [21]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=42)
X_train

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
3897,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5628,0,0,0,0,0,0,0,0,0,1,...,91,0,0,0,0,0,0,0,0,0
1756,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2346,0,0,0,0,0,1,0,0,0,0,...,1,0,0,0,0,65,23,0,0,0
2996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,0,0,0,0,0,0,0,0,0,67,...,136,120,73,0,0,0,0,0,0,0
5191,0,0,0,0,0,0,1,0,0,51,...,0,0,1,0,8,66,0,0,0,0
5226,0,0,0,0,0,0,3,1,0,0,...,112,121,121,7,0,1,0,0,0,0
5390,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
X_train = X_train/255.0
X_test = X_test/255.0

In [23]:
  # Creating Custom Dataset Class
class CustomDataset(Dataset):
    def __init__(self, features, labels):
      self.features = torch.tensor(features, dtype=torch.float32)
      self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
      return len(self.features)

    def __getitem__(self, index):
      return self.features[index], self.labels[index]

In [24]:
# Train dataset object
train_dataset = CustomDataset(X_train.values, Y_train.values)

In [25]:
# Test Dataser object
test_dataset = CustomDataset(X_test.values, Y_test.values)

In [26]:
# Creating Train and TEst Loader
train_loader = DataLoader(train_dataset, batch_size=32, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, drop_last=True)

In [27]:
# Define NN Class
class myNN(nn.Module):
    def __init__(self, input_feat):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_feat, out_features=128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)

In [28]:
learning_rate = 0.01
epoch = 100

In [29]:
model = myNN(X_train.shape[1])

In [30]:
criterion = nn.CrossEntropyLoss()

params = []
for layer in model.model:
    if isinstance(layer, nn.Module):
        params.extend(list(layer.parameters()))
optimizer = optim.SGD(params, lr=learning_rate, weight_decay=1e-4)

In [31]:
# Training Loop
for epoch in range(epoch):
    for batch_features, batch_labels in train_loader:
        # Forward Pass
        outputs = model(batch_features)

        # Loss Calc
        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()

        # Backward Pass
        loss.backward()

        optimizer.step()

In [35]:
# Testing Model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 82.77%
